# Lab 2: Learning from Confirmed Cases When Fraud Is Rare

**Time:** about 60 minutes

In Lab 1 you ignored the labels. Now use them. The investigations team confirmed about 400 fraudulent payments in
2025, out of more than 34,000. A supervised model (one that learns from examples with known answers, as on Day 2) can
learn what *those* cases have in common, which is usually far sharper than "unusual".

### Why rare cases are hard

When one class is much rarer than the other, the data is called **imbalanced**. Here only 1.2% of payments are fraud,
and that brings its own problems:

- a model can reach 98.8% accuracy by never predicting fraud, so accuracy is useless as a measure;
- a model can quietly learn to ignore the rare class, because getting it wrong costs so little during training;
- and there are two well-known traps that make a model look much better in the lab than it will be in real use.

You'll meet all of them today, and some standard ways of dealing with them.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **imbalanced data** | one class (fraud) is far rarer than the other (normal) |
| **positive / negative** | the class you're looking for (fraud = 1) / the rest (normal = 0) |
| **class weights** | telling the model that a mistake on a fraud case counts many times more than one on a normal case |
| **cost-sensitive learning** | another name for using class weights |
| **oversampling** | adding extra copies (or invented examples) of the rare class to the training data |
| **SMOTE** | Synthetic Minority Over-sampling Technique: invents new fraud examples between similar real ones |
| **undersampling** | throwing away most of the common class so the two are equal in number |
| **group split** | a train/test split that keeps all rows for one member on the same side |
| **cross-validation** | splitting the data into several parts and testing on each part in turn |
| **pipeline** | a chain of steps (scale, resample, model) run in order, always fitted on the training data only |
| **threshold** | the score above which a payment is flagged |
| **PR-AUC** | one number for how well the model ranks fraud above normal payments; higher is better |

## Where this lab fits

**What you've done so far.** In Lab 1 you ranked 34,000 payments by how unusual they are, with no labels, and found
that unusual and fraudulent aren't the same thing: rare payment types crowded the top of the list. On Day 2 you
trained supervised models on audited claims, where about 9% were irregular.

**What you'll do now.** You'll use the roughly 400 confirmed fraud payments to train a supervised model. Fraud here is
much rarer than Day 2's irregular claims (1.2%), so you'll first fall into two traps that make models look far better
than they are, then compare five ways of dealing with the imbalance, and finally choose the flagging threshold from the
cost of each kind of mistake.

**Why it matters.** A model that has learned from confirmed cases is usually much sharper than an anomaly detector at
finding fraud of a known kind. But it's also easy to fool yourself with one, and a fraud model that looks excellent in
testing and then disappoints in use wastes investigators' time and members' patience.

**What happens next.** You'll save the model for Lab 3, where it's combined with Lab 1's anomaly scores and business
rules into a single review queue.

### Theory: supervised versus unsupervised for fraud

| | Unsupervised (Lab 1) | Supervised (this lab) |
|---|---|---|
| Learns from | the payments only | the payments **and** the confirmed outcomes |
| Finds | whatever is unusual | whatever resembles past confirmed fraud |
| Good at | new schemes nobody has seen | known schemes, with far fewer false alarms |
| Weak at | telling rare-but-legitimate from wrong | anything unlike the past cases |
| Biggest risk | crowded with innocent rare payments | copying the blind spots of past investigations |

Neither is enough on its own. That's why Lab 3 uses both.

## Imports

Most of these are familiar from Day 2. The new ones:

- `GroupShuffleSplit`: a train/test split that keeps each member's payments together (Section 2);
- `StratifiedKFold` and `cross_val_score`: cross-validation (Section 4);
- `precision_recall_curve`: all the precision/recall combinations a set of scores can reach;
- **imbalanced-learn** (`imblearn`): tools for rare classes, including SMOTE and undersampling, and its own `Pipeline`
  that can include resampling steps;
- `joblib`: saves the trained model to a file for Lab 3.

In [ ]:
import os
import warnings

# Skip a slow, noisy CPU-core check on Windows 11, and hide library notices.
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from xgboost import XGBClassifier

# imbalanced-learn: tools for rare classes. Its Pipeline can include resampling steps.
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
SEED = 42        # same random choices every run

> **Where we are.** The libraries are loaded, including imbalanced-learn for the resampling methods. First, the same
> payments and features as Lab 1.

## 1. Data and features

The same payments and the same 16 features as Lab 1, built the same way. Lab 1 explains each one; here they're
written more compactly, one line each.

In [ ]:
pay = pd.read_csv("benefit_payments_2025.csv", parse_dates=["payment_date"], dtype={"member_no": "string"})

# The same 16 features as Lab 1.
X = pd.DataFrame(index=pay.index)
X["log_amount"] = np.log1p(pay["amount_tzs"])                                     # size of payment (log)
X["amount_vs_entitlement"] = pay["amount_tzs"] / pay["entitlement_tzs"]           # 1.0 = exactly what's owed
X["log_days_since_account_change"] = np.log1p(pay["account_changed_days_ago"])    # recent account change?
X["members_sharing_account"] = pay["members_sharing_account"]
X["payee_age"] = pay["payee_age"]
X["is_pension"] = pay["payment_type"].isin(["MONTHLY_PENSION", "ARREARS"]).astype(int)
X["months_since_verification"] = pay["months_since_life_verification"].fillna(0)  # months since proof of life
X["off_hours_approval"] = ((pay["approval_hour"] < 8) | (pay["approval_hour"] >= 17)).astype(int)
X["fast_payment"] = (pay["days_claim_to_payment"] < 10).astype(int)
X["mobile_money"] = (pay["account_type"] == "MOBILE_MONEY").astype(int)
X["officer_off_hours_rate"] = X["off_hours_approval"].groupby(pay["approving_officer"]).transform("mean")

# Repeats: the same amount to the same member for the same benefit within 10 days.
gap = pay.sort_values("payment_date").groupby(["member_no", "benefit_type", "amount_tzs"])["payment_date"].diff().dt.days
X["repeat_within_10d"] = (gap.reindex(pay.index) <= 10).astype(int)

# One 0/1 column per payment type (monthly pension = all zeros).
X["type_arrears"] = (pay["payment_type"] == "ARREARS").astype(int)
X["type_lump_sum"] = (pay["payment_type"] == "LUMP_SUM").astype(int)
X["type_grant"] = (pay["payment_type"] == "GRANT").astype(int)
X["type_instalment"] = (pay["payment_type"] == "INSTALMENT").astype(int)

This time the label `y` is used for training. `groups` records which member each payment belongs to; you'll need it in
Section 2.

In [ ]:
# The label, and the member each payment belongs to.
y = pay["confirmed_fraud"]
groups = pay["member_no"]

print(X.shape, f"fraud rate {y.mean():.2%}")

> **Where we are.** You have the 16 features `X`, the fraud label `y`, and `groups`, which says which member each
> payment belongs to. Before training anything, you need a test set that's honest.

### Theory: when rows aren't independent

Most splitting methods assume each row is an independent example. That's often untrue. A member's twelve monthly
pensions share the same age, account, verification history and approving officer. They're closer to *one example
repeated twelve times* than to twelve examples.

When near-identical rows land on both sides of a split, the model can score well on the test rows by **recognising**
them rather than by understanding fraud. This is a form of leakage, and it makes the test results too good.

The fix is to split by the **entity** that produces the rows (here, the member), so the test set contains only members
the model has never seen. The question to ask on any dataset is: *"what will be new when the model is used for real?"*
Split so that the same thing is new in the test set.

## 2. Trap one: the same member on both sides of the split

A ghost pensioner produces twelve fraudulent payments in a year, one a month, nearly identical. Split the payments at
random and some of those twelve land in training and the rest in testing. The model then "recognises" the test
payments because it has seen their twins, and it looks brilliant.

It's like setting an exam using questions the students already practised word for word: high marks, but they tell you
nothing about whether the students can answer *new* questions.

In real use the model will meet **new** members it has never seen. So the test set must contain only members the model
never trained on. You'll train the same model twice, with two different splits, and compare.

First, one setting both models need. XGBoost's `scale_pos_weight` tells it how much more a fraud case should count than
a normal one during training. The usual choice is the ratio of normal to fraud cases: with about 85 normal payments for
every fraud, each fraud case counts 85 times. Without it, the model could almost ignore fraud and still make very few
mistakes overall.

In [ ]:
# About 85 normal payments for every fraud. scale_pos_weight makes each fraud case count that much.
fraud_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", round(fraud_weight, 1))

### Split A: at random, by payment

The usual `train_test_split`, keeping 30% for testing. `stratify=y` keeps the same fraud rate in both parts.

The XGBoost settings are the same as Day 2: 300 shallow trees (`max_depth=4`), small steps (`learning_rate=0.05`), and
each tree sees a random 80% of the rows and columns. `predict_proba(...)[:, 1]` gives each payment's score for "fraud".

In [ ]:
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)

model_a = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=fraud_weight, eval_metric="logloss", random_state=SEED, n_jobs=4)
model_a.fit(X_train_a, y_train_a)

# PR-AUC on the test part: how well fraud is ranked above normal payments.
pr_random = average_precision_score(y_test_a, model_a.predict_proba(X_test_a)[:, 1])
print(f"PR-AUC with a random split: {pr_random:.3f}")

### Split B: by member

`GroupShuffleSplit` works like `train_test_split`, but keeps all the rows of a group (here, a member) together on one
side. It returns the **row positions** for training and testing, and `iloc` picks those rows.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)

# split() can produce several splits; we asked for one, so take the first with next().
train_rows, test_rows = next(splitter.split(X, y, groups))

X_train = X.iloc[train_rows]
y_train = y.iloc[train_rows]
X_test = X.iloc[test_rows]
y_test = y.iloc[test_rows]

# Check: no member appears on both sides. & between two sets gives the members they have in common.
members_in_train = set(groups.iloc[train_rows])
members_in_test = set(groups.iloc[test_rows])
print("members on both sides:", len(members_in_train & members_in_test))
print(f"test set: {len(X_test):,} payments, {y_test.sum()} fraud, {len(members_in_test):,} members")

Train exactly the same model on the member-level split and compare the two PR-AUC scores:

In [ ]:
model_b = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=fraud_weight, eval_metric="logloss", random_state=SEED, n_jobs=4)
model_b.fit(X_train, y_train)
pr_group = average_precision_score(y_test, model_b.predict_proba(X_test)[:, 1])

print(f"PR-AUC with a random split:     {pr_random:.3f}")
print(f"PR-AUC with members kept apart: {pr_group:.3f}")

The difference is how much the random split flattered the model. From here on, everything uses the member-level
split. The same trap exists wherever one entity produces many rows: employers and their monthly returns, members and
their claims, officers and their approvals.

> **Where we are.** The random split made the model look much better than the member split did. That difference was
> the model recognising members it had already seen. From now on, every result uses the member split.

### Theory: why a model ignores rare cases, and the three ways to stop it

During training, a model reduces its total error over all the rows. With 85 normal payments for every fraud, the
cheapest way to reduce the error is to get the normal payments right and let the frauds go. The model isn't being
lazy: it's doing exactly what it was told. You have to change what it's told. There are three ways:

| Approach | Changes | Idea |
|---|---|---|
| **Class weights** | the loss | a mistake on a fraud counts 85 times as much, so ignoring frauds is no longer cheap |
| **Oversampling** (SMOTE) | the data | add more fraud examples (invented ones) until the classes are equal |
| **Undersampling** | the data | remove normal examples until the classes are equal |

All three push the model the same way: towards taking fraud seriously. Weights do it without changing the data, which
is why they're usually the first thing to try. Resampling changes the training data, so it must only ever happen
inside the training step, never to the test data. Section 4 shows what happens when that rule is broken.

## 3. The imbalance, and five ways to handle it

You'll compare five approaches under identical conditions:

| Approach | What it does |
|---|---|
| Plain | Train as normal. The model can minimise its error by mostly ignoring fraud. |
| Class weights | Make each fraud case count as much as ~85 normal ones in the loss (**cost-sensitive learning**). |
| SMOTE | Create synthetic fraud cases by interpolating between real ones that are near each other. |
| Undersampling | Throw away most normal cases so the classes are balanced. |
| XGBoost + weights | A stronger model with the cost-sensitive setting |

The first four use **logistic regression**, a simple linear model from Day 2, so the only thing that changes between
them is how the imbalance is handled.

Three measures, all on the test members:

- **PR-AUC** (area under the precision-recall curve): the overall quality of the ranking, the better guide when
  positives are rare;
- **recall at 80% precision**: "if we insist that four out of five cases we open are real, how much of the fraud do we
  catch?" A number an investigations manager can act on;
- **precision in the top 1%**: of the 1% highest-scoring payments, the share that are fraud.

### What SMOTE actually creates

SMOTE picks a real fraud case, finds a few of the fraud cases most similar to it, and creates a new, synthetic case at a
random point on the line between them. For example, between a payment of TZS 1.2 million to an account changed 20 days
earlier and one of TZS 1.6 million to an account changed 40 days earlier, it might create TZS 1.4 million and 30 days.
Repeat until fraud and normal cases are equal in number. The new cases are plausible, but they're invented, which is why
they must never appear in the test data.

See it on the training data: `fit_resample` returns a new, larger table with the invented cases added.

In [ ]:
# See SMOTE on its own: count the classes before and after.
X_resampled, y_resampled = SMOTE(random_state=SEED).fit_resample(X_train, y_train)
print("before SMOTE:", y_train.value_counts().to_dict())
print("after SMOTE: ", y_resampled.value_counts().to_dict())

Every approach will be measured the same way, so write the measurement once. The function trains the model on the
training members, scores the test members, and works out the three measures.

**Recall at 80% precision** needs a word of explanation. As you lower the threshold, you flag more payments: recall goes
up (you catch more fraud) but precision usually goes down (more of the flagged ones are innocent).
`precision_recall_curve` lists every precision/recall pair the scores can reach. Keep the pairs where precision is at
least 0.8, and take the highest recall among them.

In [ ]:
def evaluate(name, model):
    """Train on the training members, then measure on the test members. Returns (row of results, scores)."""
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]       # each test payment's fraud score

    # Recall at 80% precision: the precision-recall curve lists every (precision, recall) pair the
    # scores can reach. Keep the pairs with precision of at least 0.8, and take the best recall among them.
    precision, recall, _ = precision_recall_curve(y_test, scores)
    good_enough = precision >= 0.8
    if good_enough.any():
        recall_at_80 = recall[good_enough].max()
    else:
        recall_at_80 = 0.0        # the model never reaches 80% precision

    # Precision in the top 1% of scores.
    ranked = pd.DataFrame({"fraud": y_test.values, "score": scores}).sort_values("score", ascending=False)
    top = ranked.head(round(0.01 * len(ranked)))

    row = {
        "approach": name,
        "PR-AUC": round(float(average_precision_score(y_test, scores)), 3),
        "recall at 80% precision": round(float(recall_at_80), 3),
        "precision top 1%": round(float(top["fraud"].mean()), 3),
    }
    return row, scores

Now the five approaches, one cell each. Each is a **pipeline**: the steps run in order, and they're all fitted on the
training data only. Scaling comes first because logistic regression works best when features are on similar scales.

**1. Plain:** no special treatment of the imbalance.

In [ ]:
# 1. Plain logistic regression.
plain = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000)),      # max_iter: allow enough steps to finish learning
])
row_plain, p_plain = evaluate("Plain logistic regression", plain)
row_plain

**2. Class weights:** `class_weight="balanced"` makes each class count in inverse proportion to how common it is, so
the rare fraud cases count far more. Nothing is added or removed from the data.

In [ ]:
# 2. Class weights: "balanced" weights each class by how rare it is.
weighted = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced")),
])
row_weighted, p_weighted = evaluate("Logistic + class weights", weighted)
row_weighted

**3. SMOTE:** invent extra fraud cases before training. This needs imbalanced-learn's own pipeline (`ImbPipeline`),
because scikit-learn's ordinary pipeline has no resampling steps. Importantly, the resampling only happens while
training; when the model scores the test set, no invented cases are added.

In [ ]:
# 3. SMOTE. imblearn's Pipeline applies SMOTE while training only, never when scoring the test set.
smote = ImbPipeline([
    ("scale", StandardScaler()),
    ("smote", SMOTE(random_state=SEED)),
    ("model", LogisticRegression(max_iter=3000)),
])
row_smote, p_smote = evaluate("Logistic + SMOTE", smote)
row_smote

**4. Undersampling:** the opposite approach. Keep all the fraud cases, and keep only an equal number of randomly chosen
normal ones; throw the rest of the normal cases away for training.

In [ ]:
# 4. Undersampling: keep all fraud cases and an equal number of randomly chosen normal ones.
under = ImbPipeline([
    ("scale", StandardScaler()),
    ("under", RandomUnderSampler(random_state=SEED)),
    ("model", LogisticRegression(max_iter=3000)),
])
row_under, p_under = evaluate("Logistic + undersampling", under)
row_under

**5. XGBoost with weights:** a stronger model (trees, which can combine features) with the same weighting idea as
approach 2. Trees don't need scaling, so there's no pipeline. Then put all five results in one table.

In [ ]:
# 5. XGBoost with the fraud weight.
xgb_weighted = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=fraud_weight, eval_metric="logloss", random_state=SEED, n_jobs=4)
row_xgb, p_xgb = evaluate("XGBoost + class weights", xgb_weighted)

comparison = pd.DataFrame([row_plain, row_weighted, row_smote, row_under, row_xgb]).set_index("approach")
comparison.round(3)

A few things usually show up in a table like this:

- **Rebalancing changes the probabilities more than the ranking.** Class weights, SMOTE (Synthetic Minority
  Over-sampling Technique) and undersampling often give similar PR-AUC to each other. What they mainly change is where
  the scores sit, which matters when you pick a threshold. It matters less when you simply review the top of the list.
- **Undersampling throws away information.** With 34,000 normal payments and a few hundred frauds, discarding most
  of the normal ones leaves the model knowing less about what normal looks like.
- **The model matters more than the resampling trick.** A tree model that can combine features ("shared account
  *and* recent change *and* elderly payee") usually beats any rebalanced linear model.

**Your turn (1).** Try XGBoost (eXtreme Gradient Boosting) with SMOTE (`ImbPipeline([("smote", SMOTE(random_state=SEED)), ("model", ...)])`,
using an XGBoost *without* `scale_pos_weight`). Does SMOTE help a model that already handles the imbalance with
weights?

In [ ]:
# Your code here

> **Where we are.** You've compared five approaches under identical conditions. The model mattered more than the
> resampling trick: a tree model that can combine features usually comes out ahead. Now the second trap, which
> catches people using SMOTE.

## 4. Trap two: SMOTE before the split

SMOTE is often applied in the wrong place. If you oversample the **whole** dataset first and then cross-validate,
synthetic fraud cases built from a real case end up in the test folds next to their "parent" in the training folds.
The model is tested on near-copies of what it learned from.

### A quick reminder of cross-validation

Instead of one train/test split, **cross-validation** splits the data into 5 parts (called **folds**). It trains on 4
and tests on the fifth, then repeats so each part gets a turn as the test part, giving 5 scores. Their average is a
steadier estimate than any single split. `StratifiedKFold` makes sure each fold has the same fraud rate, and
`cross_val_score` does the training and testing for you.

Compare the wrong way and the right way, on the training data:

In [ ]:
# The same 5 folds for both, each with the same fraud rate.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# WRONG: scale and oversample all the training data first, then cross-validate on the result.
X_scaled = StandardScaler().fit_transform(X_train)
X_oversampled, y_oversampled = SMOTE(random_state=SEED).fit_resample(X_scaled, y_train)

wrong_scores = cross_val_score(LogisticRegression(max_iter=3000), X_oversampled, y_oversampled,
                               cv=folds, scoring="average_precision")      # average_precision = PR-AUC
wrong = wrong_scores.mean()

The right way puts SMOTE **inside** the pipeline. Then, in each of the 5 rounds, the pipeline is refitted on that
round's 4 training folds only, so SMOTE never sees the test fold and no invented case is ever tested on.

In [ ]:
# RIGHT: SMOTE inside the pipeline, so in each fold it only ever sees that fold's training part.
right_pipeline = ImbPipeline([
    ("scale", StandardScaler()),
    ("smote", SMOTE(random_state=SEED)),
    ("model", LogisticRegression(max_iter=3000)),
])
right_scores = cross_val_score(right_pipeline, X_train, y_train, cv=folds, scoring="average_precision")
right = right_scores.mean()

print(f"cross-validated PR-AUC, SMOTE before splitting: {wrong:.3f}   <- looks wonderful, means nothing")
print(f"cross-validated PR-AUC, SMOTE inside the folds: {right:.3f}")

The wrong way also changes what's being measured: its test folds are half synthetic, so the "precision" it reports is
precision on a world where fraud is as common as normal payments. The rule: **any step that learns from the data
(scaling, imputing, resampling, feature selection) goes inside the pipeline, so it's refitted on each training fold.**
That's the same principle as Day 2's preprocessing pipelines.

> **Where we are.** Both traps are behind you: split by member, and put every step that learns from data inside the
> pipeline. You have an honestly tested model whose scores rank fraud well. The last decision is where to draw the
> line.

### Theory: the cost of each kind of mistake

A flagging rule makes two kinds of mistake, and they don't cost the same:

| Mistake | Name | What it costs the Fund |
|---|---|---|
| flagging an innocent payment | false positive | officer time, and a delayed payment for a member who did nothing wrong |
| missing a fraudulent payment | false negative | the money paid, and for a ghost pension, every future month too |

Every possible threshold gives a different mix of the two. The **expected cost** of a threshold adds them up:

```
total cost = (number flagged × cost of one review) + (sum of fraud amounts missed × months of loss)
```

Pick the threshold with the lowest total. This turns an arbitrary technical choice (0.5) into a business decision that
can be explained, argued over and changed when the costs change. It also makes the assumptions visible: if someone
disagrees with "TZS 40,000 per review", they can change that number and see the effect.

## 5. Choosing a threshold by cost, not by habit

A model gives scores between 0 and 1. Someone has to decide above which score a payment is held for review. The default
of 0.5 has no business meaning: it's just the middle of the scale. A better approach is to put a price on each kind of
mistake:

- **Reviewing a payment** costs officer time: say **TZS 40,000** per case (about half a day, plus the member's
  inconvenience if a legitimate payment is delayed).
- **Missing a fraudulent payment** costs the payment itself, and for a ghost pension it costs every future month
  too. Say **12 × the amount** for monthly pensions and **1 × the amount** for one-off payments.

Then pick the threshold with the lowest total cost.

First, work out for every test payment what it would cost if it were fraud and you missed it. `np.where(condition, a,
b)` gives `a` where the condition is true and `b` where it isn't: 12 months for monthly pensions, 1 for everything else.

In [ ]:
REVIEW_COST = 40_000      # TZS per review (the _ just makes the number easier to read)

# The test payments, and what each would cost if it were fraud and we missed it.
test_payments = pay.iloc[test_rows]
months_lost = np.where(test_payments["payment_type"] == "MONTHLY_PENSION", 12, 1)
loss_if_missed = months_lost * test_payments["amount_tzs"].to_numpy()

is_fraud = y_test.to_numpy() == 1      # True for the test payments that are fraud

Now try 99 thresholds, from 0.01 to 0.99. For each one:

- every payment scoring at or above it is **flagged** and reviewed, at TZS 40,000 each;
- every fraud payment *below* it is **missed**, and costs its `loss_if_missed`. `~flagged & is_fraud` means "not
  flagged *and* fraud".

`np.linspace(0.01, 0.99, 99)` makes 99 evenly spaced numbers between 0.01 and 0.99. The scores come from the XGBoost
model (`p_xgb`).

In [ ]:
# Try 99 thresholds from 0.01 to 0.99 and work out the cost of each.
rows = []
for threshold in np.linspace(0.01, 0.99, 99):
    flagged = p_xgb >= threshold

    review_cost = flagged.sum() * REVIEW_COST                    # every flagged payment is reviewed
    missed_cost = loss_if_missed[~flagged & is_fraud].sum()      # fraud that wasn't flagged

    rows.append({
        "threshold": threshold,
        "flagged": int(flagged.sum()),
        "review_cost": review_cost,
        "missed_fraud_cost": missed_cost,
        "total_cost": review_cost + missed_cost,
    })

costs = pd.DataFrame(rows)
costs.head()

Plot the two costs and their total against the threshold. A low threshold flags many payments (high review cost, little
missed fraud); a high threshold flags few (low review cost, lots of missed fraud). The best threshold is where the
black total line is lowest, marked with a dashed line.

In [ ]:
# The threshold with the lowest total cost.
best = costs.loc[costs["total_cost"].idxmin()]

plt.plot(costs["threshold"], costs["review_cost"] / 1e6, label="cost of reviews", color="#2a78d6")
plt.plot(costs["threshold"], costs["missed_fraud_cost"] / 1e6, label="cost of missed fraud", color="#eb6834")
plt.plot(costs["threshold"], costs["total_cost"] / 1e6, label="total", color="black", lw=2)
plt.axvline(best["threshold"], color="grey", ls="--")      # the cheapest threshold
plt.xlabel("score threshold")
plt.ylabel("TZS millions")
plt.title("Choosing a threshold by cost", loc="left")
plt.legend()
plt.show()

print(f"lowest total cost at threshold {best['threshold']:.2f}: {int(best['flagged'])} payments reviewed, "
      f"total cost TZS {best['total_cost'] / 1e6:,.1f}m")

Change the assumptions and the best threshold moves. That's the point: the threshold is a **business decision
expressed in numbers**, and it should be agreed with the investigations manager and finance, written down, and
revisited when costs change. In Lab 3 it goes into a configuration file, not into the code.

**Your turn (2).** Suppose a new rule means every flagged payment is held until reviewed, which upsets members and
generates complaints. Finance estimates the real cost of each review at TZS 150,000. Rerun the cost analysis. How
many payments would you review now, and how much fraud would you accept missing?

In [ ]:
# Your code here

> **Where we are.** The cost analysis has given you a threshold, and you've seen how it moves when the cost of a
> review changes. The model is ready to hand on.

## 6. Save the model for Lab 3

The test split did its job: it told you how well the model works on new members. For the model you'll actually use,
retrain on **all** the 2025 payments, so it learns from every confirmed case.

`joblib.dump` saves Python objects to a file. Save the model *together with the list of feature names*: Lab 3 must
build its features in exactly the same order, and saving the list with the model makes that easy to check.

In [ ]:
# Retrain on all 2025 payments, then save the model together with its list of features.
final = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                      scale_pos_weight=fraud_weight, eval_metric="logloss", random_state=SEED, n_jobs=4)
final.fit(X, y)

joblib.dump({"model": final, "features": list(X.columns)}, "fraud_model.joblib")
print("saved fraud_model.joblib")

---

## Looking back, looking ahead

**What you did in this lab.** You trained supervised fraud models on very rare labels: split by member to avoid
recognising members, compared plain training, class weights, SMOTE, undersampling and weighted XGBoost under identical
conditions, saw how SMOTE before the split inflates results, picked a threshold by putting prices on each kind of
mistake, and saved a final model with its feature list.

**The ideas to carry forward.**

- Split by the thing that will be new in real use (here, members).
- Anything that learns from data, resampling included, goes inside the pipeline.
- A stronger model usually beats a resampling trick.
- The threshold is a business decision; set it from costs, and write it down.

**What happens next.** Lab 3 brings it all together: Lab 1's anomaly scores, this lab's fraud model, and rules from the
business, combined into one score and a queue the investigations team can work through.

## Suggested answers

**(1)** XGBoost with SMOTE. Use XGBoost without the weight, so SMOTE is the only thing handling the imbalance, and
compare with approach 5:

In [ ]:
# XGBoost WITHOUT scale_pos_weight, because SMOTE is doing the balancing now.
xgb_plain = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          eval_metric="logloss", random_state=SEED, n_jobs=4)
xgb_smote = ImbPipeline([
    ("smote", SMOTE(random_state=SEED)),
    ("model", xgb_plain),
])
row_xgb_smote, _ = evaluate("XGBoost + SMOTE", xgb_smote)      # _ : we don't need the scores

pd.DataFrame([row_xgb, row_xgb_smote]).set_index("approach").round(3)

Typically, little or no gain. Weights already tell a boosted model to take the rare class seriously, and synthetic
points interpolated between real frauds add little the trees couldn't learn from the real cases. SMOTE is most useful
for simple models on small datasets. It isn't a default step.

**(2)** Higher review cost. The same loop with the new cost, also counting how many fraud payments are missed:

In [ ]:
REVIEW_COST = 150_000

rows = []
for threshold in np.linspace(0.01, 0.99, 99):
    flagged = p_xgb >= threshold
    review_cost = flagged.sum() * REVIEW_COST
    missed_cost = loss_if_missed[~flagged & is_fraud].sum()
    rows.append({
        "threshold": threshold,
        "flagged": int(flagged.sum()),
        "total_cost": review_cost + missed_cost,
        "fraud_missed": int((~flagged & is_fraud).sum()),       # fraud payments below the threshold
    })

costs_150 = pd.DataFrame(rows)
costs_150.loc[costs_150["total_cost"].idxmin()]       # the cheapest threshold